In [10]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

verbose = 0

data_path = "/home/chenzihao/workspace/cc2cc_test5/validate/ccdft_atom-1-2008676_g2.csv"
# basis_args = data_path.split("/")[-1].split("_")[1]
basis_args = "cc-pVDZ"
print(basis_args)

with open("../cc2cc/utils/g2.json") as f:
    json_data = json.load(f)

data = pd.read_csv(data_path)

data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

data_name = []
data_atomic_energy_dft = []
data_atomic_energy_ai = []
data_atomic_dft_ele = []
data_atomic_scf_ele = []
data_atomic_dft_dip = []
data_atomic_scf_dip = []


for i_name in data["name"]:
    if i_name not in json_data["reaction-atomic-energy"]:
        continue
    systems_list = json_data["reaction-atomic-energy"][i_name]["systems"]
    stoichiometry_list = json_data["reaction-atomic-energy"][i_name]["stoichiometry"]

    atomic_energy_dft = 0
    atomic_energy_ai = 0
    for i in range(len(systems_list)):
        atomic_energy_dft += data[data["name"] == systems_list[i]][
            "error_dft_ene"
        ].values[0] * int(stoichiometry_list[i])
        if verbose:
            print(
                data[data["name"] == systems_list[i]]["error_dft_ene"].values[0],
                int(stoichiometry_list[i]),
                systems_list[i],
            )
        atomic_energy_ai += data[data["name"] == systems_list[i]][
            "error_scf_ene"
        ].values[0] * int(stoichiometry_list[i])
    if verbose:
        print(atomic_energy_dft, "\n")
    data_atomic_energy_dft.append(atomic_energy_dft)
    data_atomic_energy_ai.append(atomic_energy_ai)
    data_name.append(i_name)

    data_atomic_dft_ele.append(
        data.loc[data["name"] == i_name, "error_dft_ele"].values[0]
    )
    data_atomic_scf_ele.append(
        data.loc[data["name"] == i_name, "error_scf_ele"].values[0]
    )
    data_atomic_dft_dip.append(
        data.loc[data["name"] == i_name, "error_dft_dip"].values[0]
    )
    data_atomic_scf_dip.append(
        data.loc[data["name"] == i_name, "error_scf_dip"].values[0]
    )

data_name = np.array(data_name)
data_atomic_energy_dft = np.array(data_atomic_energy_dft)
data_atomic_energy_ai = np.array(data_atomic_energy_ai)
data_atomic_dft_ele = np.array(data_atomic_dft_ele)
data_atomic_scf_ele = np.array(data_atomic_scf_ele)

print(np.mean(np.abs(data_atomic_energy_dft)))
print(np.mean(np.abs(data_atomic_energy_ai)))

sorted_indices = np.argsort(np.abs(data_atomic_energy_dft))[::-1][:10]
sorted_data_atomic_energy_dft = data_name[sorted_indices]
print(
    "dft",
    np.array(
        [
            sorted_data_atomic_energy_dft,
            np.array(data_atomic_energy_dft)[sorted_indices],
        ]
    ).T,
)
sorted_indices = np.argsort(np.abs(data_atomic_energy_ai))[::-1][:10]
sorted_data_atomic_energy_ai = data_name[sorted_indices]
print(
    "ai",
    np.array(
        [
            sorted_data_atomic_energy_ai,
            np.array(data_atomic_energy_ai)[sorted_indices],
        ]
    ).T,
)

print("dft_ele", np.mean(np.abs(data_atomic_dft_ele)))
print("scf_ele", np.mean(np.abs(data_atomic_scf_ele)))
print("dft_dip", np.mean(np.abs(data_atomic_dft_dip)))
print("scf_dip", np.mean(np.abs(data_atomic_scf_dip)))

cc-pVDZ
13.342593352794529
2.360841386312937
dft [['c-n2h2' '-39.544582136019045']
 ['b2h6' '-30.922330789257103']
 ['ch4' '-30.07762995627222']
 ['c-hcoh' '-25.426255652407992']
 ['bn3pi' '-25.256047433249336']
 ['nh3' '-23.45360229632883']
 ['ph3' '-20.047835927154047']
 ['nh2' '-19.72056746992323']
 ['bn' '-18.212819786851867']
 ['ch3' '-16.267201867465445']]
ai [['ph3' '-14.58884860556894']
 ['h2s' '6.1816240757660985']
 ['ch4' '-6.118887497432759']
 ['sih' '-5.020242777905627']
 ['be2' '-4.775333274056348']
 ['sih4' '-4.063139458966724']
 ['b2h6' '3.9613037790277']
 ['c2' '-3.8147356503210244']
 ['alh3' '-3.04412449300902']
 ['hf' '2.939150555420145']]
dft_ele 0.10943423896077108
scf_ele 0.11418334108951077
dft_dip 0.025291992920079498
scf_dip 0.02920481116105507


In [26]:
1345.4299291869957 / 3 - 642.8817092614942 + 287.5278185261237 * 2 / 3

-2.7198538484131234